# NeuralAtmosphereOperator — khám phá và thiết kế dữ liệu khí quyển

> **Phạm vi của notebook.** Tài liệu này chỉ nói về **dữ liệu**: nguồn ERA5/WeatherBench2, ý nghĩa của lưới latitude–longitude, 26 channels được chọn, cách giảm độ phân giải từ $0.25^\circ$ xuống $0.5^\circ$, tensor cuối cùng và các kiểm tra cần thực hiện trước khi train. Kiến trúc model, loss, normalization, training pipeline và evaluation không thuộc phạm vi ở đây.

Mục tiêu là trả lời rõ bốn câu hỏi:

1. Mỗi sample mô tả trạng thái khí quyển nào?
2. 26 channels gồm những đại lượng vật lý nào?
3. Lưới $361\times720$ được tạo từ lưới $721\times1440$ như thế nào?
4. Làm sao biết file tải về đúng trước khi dùng nó cho một lần train tốn kém?

> **Quy ước thuật ngữ.** Các thuật ngữ quen thuộc như *channel*, *grid*, *reanalysis*, *pressure level*, *regridding*, *aliasing* và *Zarr* được giữ bằng tiếng Anh khi dịch sang tiếng Việt làm câu văn gượng ép. Mỗi khái niệm đều được giải thích khi xuất hiện lần đầu.

## 1. Dữ liệu đến từ đâu?

Nguồn của dự án là bộ **ERA5** đã được WeatherBench2 chuẩn bị sẵn trên Google Cloud Storage:

> gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr

Tên của dataset cho biết các đặc điểm chính:

- **1959–2023:** khoảng thời gian mà archive nguồn cung cấp; dự án chỉ chọn một đoạn con cần thiết.
- **0.25°:** khoảng cách góc giữa hai grid points kề nhau ở dataset nguồn.
- **wb13:** dữ liệu trên 13 pressure levels chuẩn của WeatherBench.
- **6h:** một trạng thái khí quyển sau mỗi 6 giờ.
- **1440 × 721:** 1440 kinh độ và 721 vĩ độ, bao gồm cả hai cực.
- **with derived variables:** ngoài các trường ERA5 cơ bản còn có một số biến được WeatherBench2 suy ra trước.

ERA5 là **reanalysis**, không phải một tập ảnh vệ tinh thô và cũng không phải dự báo của model khác. Reanalysis kết hợp mô hình vật lý, quan trắc và data assimilation để tái dựng một trạng thái khí quyển nhất quán theo thời gian. Vì vậy mỗi timestep có thể được xem như một ước lượng tốt của trạng thái thật, nhưng không phải phép đo hoàn hảo ở mọi grid point.

Nguồn tham khảo: [WeatherBench2 Data Guide](https://weatherbench2.readthedocs.io/en/latest/data-guide.html), [ERA5 — Hersbach et al., 2020](https://doi.org/10.1002/qj.3803), [WeatherBench2 — Rasp et al., 2024](https://arxiv.org/abs/2308.15560).

## 2. Một timestep trông như thế nào?

Tại một thời điểm $t$, dữ liệu cuối cùng là tensor

$$
\mathbf{x}_t\in\mathbb{R}^{26\times361\times720}.
$$

Ba trục lần lượt là:

- **channel:** 26 trường vật lý được liệt kê ở các phần sau;
- **latitude:** 361 hàng từ $90^\circ$ Bắc đến $90^\circ$ Nam, bước $0.5^\circ$;
- **longitude:** 720 cột từ $0^\circ$ đến $359.5^\circ$, bước $0.5^\circ$.

Dữ liệu có cadence 6 giờ, tương ứng với các mốc 00, 06, 12 và 18 UTC mỗi ngày. Config hiện tại cũng dùng stride 6 giờ, vì vậy **không bỏ bớt timestep** và không nội suy theo thời gian.

Có thể hình dung một timestep như 26 bản đồ toàn cầu xếp chồng lên nhau. Mọi bản đồ dùng chung grid và timestamp, nhưng giá trị cùng đơn vị vật lý của riêng channel đó.

## 3. Đọc lưới latitude–longitude đúng cách

Latitude $\varphi$ đo vị trí Bắc–Nam, longitude $\lambda$ đo vị trí Đông–Tây. Trên grid $0.5^\circ$:

- khoảng cách Bắc–Nam giữa hai hàng xấp xỉ $55.6$ km;
- khoảng cách Đông–Tây phụ thuộc latitude và xấp xỉ $55.6\cos\varphi$ km.

Do đó grid latitude–longitude **không phải equal-area grid**. Một ô gần xích đạo có diện tích lớn hơn nhiều so với một ô gần cực. Với bán kính Trái Đất $R$, diện tích ô có biên $[\varphi_1,\varphi_2]$ và $[\lambda_1,\lambda_2]$ là

$$
A=R^2(\lambda_2-\lambda_1)\left(\sin\varphi_2-\sin\varphi_1\right),
$$

trong đó các góc được tính bằng radian. Hệ số $\sin\varphi$ này là lý do phép regridding và các phép trung bình toàn cầu không được coi mọi grid point có trọng số bằng nhau.

Longitude là trục **periodic**: sau $359.5^\circ$ là $0^\circ$, không có một đường biên vật lý tại kinh tuyến gốc. Latitude thì không periodic; $90^\circ$ Bắc và $90^\circ$ Nam là hai cực.

## 4. Pressure level là gì?

Khí quyển ba chiều thường được biểu diễn trên các mặt có cùng áp suất thay vì các mặt có cùng độ cao. Đơn vị thường dùng là hectopascal:

$$1\ \mathrm{hPa}=100\ \mathrm{Pa}. $$

Áp suất càng thấp thì mặt đó nhìn chung càng cao trong khí quyển. Tuy nhiên pressure level **không phải một độ cao cố định**: độ cao hình học của mặt 500 hPa thay đổi theo nhiệt độ và trạng thái khí quyển.

| Pressure level | Cách hiểu gần đúng | Vai trò thường gặp |
|---:|---|---|
| 1000 hPa | sát mặt đất ở nơi địa hình thấp | gió, ẩm và geopotential tầng thấp |
| 850 hPa | lower troposphere | khối khí, vận chuyển nhiệt và hơi nước |
| 500 hPa | mid-troposphere | rãnh, sống khí áp và circulation quy mô lớn |
| 250 hPa | upper troposphere | jet stream và dòng dẫn hướng |
| 100 hPa | gần tropopause/lower stratosphere | cấu trúc nhiệt tầng cao |
| 50 hPa | lower stratosphere | circulation tầng bình lưu |

Bảng chỉ nhằm tạo trực giác. Quan hệ pressure–height thay đổi theo không gian và thời gian, nên không nên gán cho mỗi pressure level một độ cao tuyệt đối.

> **Lưu ý ở vùng núi cao.** Một pressure surface như 1000 hPa có thể nằm thấp hơn mặt đất địa phương. Giá trị pressure-level của reanalysis tại những vị trí đó chịu ảnh hưởng của cách mô hình/data assimilation biểu diễn hoặc ngoại suy vùng dưới bề mặt; không nên diễn giải nó như một phép đo không khí thật ở dưới địa hình.

# Phần I — 26 channels được đưa vào model

## 5. Sáu surface channels

Surface channel là trường hai chiều, không có trục pressure level trong archive. Dự án chọn sáu trường sau:

| Channel | Tên trong WeatherBench2 | Đơn vị | Ý nghĩa |
|---:|---|---|---|
| 1 | 10m_u_component_of_wind | $\mathrm{m\,s^{-1}}$ | thành phần gió hướng Đông–Tây ở 10 m; dương theo hướng Đông |
| 2 | 10m_v_component_of_wind | $\mathrm{m\,s^{-1}}$ | thành phần gió hướng Bắc–Nam ở 10 m; dương theo hướng Bắc |
| 3 | 2m_temperature | K | nhiệt độ không khí ở 2 m |
| 4 | surface_pressure | Pa | áp suất thật tại bề mặt địa hình |
| 5 | mean_sea_level_pressure | Pa | áp suất đã quy về mực nước biển, hữu ích để nhận biết hệ thống synoptic |
| 6 | total_column_water_vapour | $\mathrm{kg\,m^{-2}}$ | tổng lượng hơi nước tích phân theo cột khí quyển |

Hai thành phần gió không phải speed và direction. Tốc độ gió 10 m có thể suy ra bằng

$$
V_{10}=\sqrt{u_{10}^2+v_{10}^2}.
$$

Surface pressure và mean sea-level pressure cũng không trùng nhau: trường thứ nhất chịu ảnh hưởng trực tiếp của độ cao địa hình; trường thứ hai thuận tiện hơn khi so sánh các hệ áp suất trên những vùng có độ cao khác nhau.

## 6. Hai mươi pressure-level channels

Một pressure-level variable có thể tồn tại ở cả 13 levels của dataset nguồn, nhưng dự án chỉ chọn một số level có chủ đích. Mỗi cặp **variable @ pressure level** trở thành một channel độc lập.

| Channels | Variable | Levels được chọn | Số channel | Ý nghĩa vật lý |
|---:|---|---|---:|---|
| 7–11 | geopotential | 1000, 850, 500, 250, 50 hPa | 5 | cấu trúc độ cao của các mặt đẳng áp và trường khối lượng |
| 12–15 | u_component_of_wind | 1000, 850, 500, 250 hPa | 4 | chuyển động Đông–Tây từ tầng thấp đến upper troposphere |
| 16–19 | v_component_of_wind | 1000, 850, 500, 250 hPa | 4 | chuyển động Bắc–Nam trên cùng các tầng |
| 20–23 | temperature | 850, 500, 250, 100 hPa | 4 | cấu trúc nhiệt theo chiều thẳng đứng |
| 24–25 | specific_humidity | 1000, 850 hPa | 2 | hơi nước tập trung ở lower troposphere |
| 26 | relative_humidity | 500 hPa | 1 | độ ẩm tương đối ở mid-troposphere |

Tổng số channels là

$$
6+(5+4+4+4+2+1)=26.
$$

**Geopotential** $\Phi$ có đơn vị $\mathrm{m^2\,s^{-2}}$. Nếu chỉ cần trực giác về độ cao geopotential, có thể dùng

$$z\approx\frac{\Phi}{g_0},\qquad g_0\approx9.80665\ \mathrm{m\,s^{-2}}. $$

**Specific humidity** là khối lượng hơi nước trên tổng khối lượng không khí ẩm, thường có đơn vị $\mathrm{kg\,kg^{-1}}$. **Relative humidity** là một tỉ số không thứ nguyên phụ thuộc cả lượng hơi nước lẫn nhiệt độ; hai đại lượng này không thể thay thế trực tiếp cho nhau.

## 7. Channel order chính xác

Channel order là một phần của data contract, không phải chi tiết trình bày. Model phải đọc đúng thứ tự đã dùng khi tạo dataset và checkpoint.

| Index | Channel | Index | Channel |
|---:|---|---:|---|
| 1 | 10m_u_component_of_wind | 14 | u_component_of_wind@500hPa |
| 2 | 10m_v_component_of_wind | 15 | u_component_of_wind@250hPa |
| 3 | 2m_temperature | 16 | v_component_of_wind@1000hPa |
| 4 | surface_pressure | 17 | v_component_of_wind@850hPa |
| 5 | mean_sea_level_pressure | 18 | v_component_of_wind@500hPa |
| 6 | total_column_water_vapour | 19 | v_component_of_wind@250hPa |
| 7 | geopotential@1000hPa | 20 | temperature@850hPa |
| 8 | geopotential@850hPa | 21 | temperature@500hPa |
| 9 | geopotential@500hPa | 22 | temperature@250hPa |
| 10 | geopotential@250hPa | 23 | temperature@100hPa |
| 11 | geopotential@50hPa | 24 | specific_humidity@1000hPa |
| 12 | u_component_of_wind@1000hPa | 25 | specific_humidity@850hPa |
| 13 | u_component_of_wind@850hPa | 26 | relative_humidity@500hPa |

Output Zarr lưu kèm channel name, units, long name, source variable và pressure level. Nhờ vậy có thể kiểm tra thứ tự bằng metadata thay vì dựa vào trí nhớ hoặc tên file.

## 8. “Tổng số channels” trong archive là bao nhiêu?

Cần phân biệt **variable** và **channel**. Một variable hai chiều như 2m_temperature tạo một channel. Một variable bốn chiều như temperature có thêm trục pressure level; nếu lấy đủ 13 levels thì riêng variable đó tạo 13 channels.

Metadata của đúng WeatherBench2 archive mà config đang trỏ tới có cấu trúc sau, sau khi bỏ bốn coordinate arrays time, level, latitude và longitude:

| Nhóm | Số variables | Số maps sau khi trải level | Có thay đổi theo thời gian? |
|---|---:|---:|---|
| trường bề mặt hoặc trường đơn tầng | 34 | 34 | có |
| trường trên 13 pressure levels | 15 | $15\times13=195$ | có |
| trường static như địa hình, đất–biển, vegetation | 13 | 13 | không |
| **Tổng** | **62** | **242** | hỗn hợp |

Nếu chỉ đếm các trường động có thể làm forecast state/target thì archive có tối đa

$$
34+15\times13=229\text{ channels}.
$$

Mười ba static fields có thể được thêm làm auxiliary inputs, nhưng không hợp lý khi dự báo chúng ở mỗi bước vì chúng không tiến hóa theo thời gian. Vì vậy cách nói chính xác là: baseline chọn **26 trong 229 dynamic channels khả dụng**, đồng thời không dùng 13 static fields. Tỉ lệ $26/229\approx11.4\%$ không có nghĩa model chỉ nhận 11.4% “thông tin khí quyển”; nhiều channels còn lại là biến dẫn xuất, biến gần trùng nhau, flux/accumulation hoặc các lát cắt level không cần thiết cho baseline.

## 8.1. Năm tiêu chí chọn 26 channels

Bộ channels được chọn theo một **engineering trade-off** giữa độ đầy đủ vật lý, khả năng học, storage, I/O và compute. Nó không phải tuyên bố rằng khí quyển thật chỉ có 26 degrees of freedom.

### 1. Giữ các nhóm trạng thái vật lý cốt lõi

- Gió $u,v$ biểu diễn chuyển động ngang và advection.
- Surface pressure, mean sea-level pressure và geopotential biểu diễn trường khối lượng cùng cấu trúc các mặt đẳng áp.
- Temperature biểu diễn trạng thái nhiệt động lực học.
- Specific humidity, relative humidity và total column water vapour biểu diễn phân bố hơi nước.

Như vậy 26 channels không chỉ tập trung vào một đại lượng dễ dự báo mà bao phủ bốn nhóm liên kết với nhau: **dynamics, mass/pressure, temperature và moisture**.

### 2. Giữ đại diện theo chiều thẳng đứng

Không lấy đủ 13 levels cho mọi variable. Thay vào đó, các levels được chọn để quan sát lower troposphere, mid-troposphere, upper troposphere và một phần lower stratosphere:

- 1000/850 hPa: boundary layer và lower-tropospheric transport;
- 500 hPa: circulation quy mô lớn ở mid-troposphere;
- 250 hPa: upper-level flow và jet stream;
- 100/50 hPa: cấu trúc nhiệt hoặc geopotential ở tầng cao.

Mỗi variable có level selection riêng vì thông tin hữu ích của chúng không phân bố giống nhau. Moisture tập trung mạnh ở tầng thấp, trong khi geopotential và temperature cần vertical coverage rộng hơn.

### 3. Tránh trả chi phí hai lần cho thông tin gần trùng nhau

Archive chứa nhiều derived variables như wind speed, vorticity, divergence, geostrophic/ageostrophic wind speed, eddy kinetic energy, integrated vapor transport và lapse rate. Chúng có ích cho phân tích, nhưng phần lớn được suy ra hoàn toàn hoặc gần hoàn toàn từ gió, temperature, pressure và moisture đã chọn.

Ví dụ, khi đã có $u$ và $v$, wind speed thỏa

$$V=\sqrt{u^2+v^2}.$$

Thêm cả $V$ làm một target riêng làm tăng storage, I/O và loss terms mà không đưa vào một degree of freedom độc lập tương ứng. Baseline ưu tiên các trường cơ sở hơn các bản tóm tắt dẫn xuất.

### 4. Giữ ý nghĩa one-step state nhất quán

Precipitation 6/12/24 giờ và nhiều radiation/heat flux là đại lượng tích lũy hoặc trung bình trên một time window. Chúng không có cùng semantics với một trạng thái tức thời tại timestamp $t$. Đưa chúng vào cùng autoregressive state đòi hỏi định nghĩa lại target window, cách nối rollout và loss; nếu xử lý không cẩn thận rất dễ gây lệch thời gian hoặc leakage.

Các biến land/ocean như soil moisture, sea-surface temperature, sea ice và vegetation lại tiến hóa trên time scales khác và thường cần coupling hoặc forcing riêng. Loại chúng khỏi baseline giúp bài toán one-step atmosphere-to-atmosphere có ranh giới rõ ràng hơn.

### 5. Giữ chi phí phù hợp với dự án

Ở cùng grid và khoảng thời gian, storage/transfer của tensor động tăng gần tuyến tính theo số channels. Nếu lưu cả 229 dynamic channels dưới dạng float32 ở $0.5^\circ$, logical uncompressed size sẽ xấp xỉ

$$
952.9\ \mathrm{GB}\times\frac{229}{26}\approx8.40\ \mathrm{TB},
$$

so với 952.9 GB của 26 channels. Compression có thay đổi dung lượng vật lý nhưng không xóa chênh lệch I/O, decompression và memory bandwidth. Số input/output parameters cũng tăng theo channel count, dù phần spectral core của SFNO vẫn chủ yếu do embedding dimension và số layers quyết định.

## 8.2. Những gì bị loại và giới hạn của quyết định này

| Nhóm không được chọn | Ví dụ trong archive | Lý do chưa đưa vào baseline |
|---|---|---|
| derived dynamics | wind_speed, vorticity, divergence, geostrophic_wind_speed | phần lớn thông tin có thể suy ra từ $u,v$ và pressure/geopotential; tránh redundancy |
| precipitation/flux | precipitation 6/12/24 h, radiation và surface heat fluxes | window semantics khác state tức thời; cần target/loss chuyên biệt |
| land/ocean state | soil moisture, SST, sea ice, snow depth | time scale và coupling khác khí quyển tự hồi quy |
| static fields | surface geopotential, land–sea mask, vegetation, orography statistics | có thể làm auxiliary forcing nhưng baseline hiện không dùng forcing |
| pressure levels còn lại | 925, 700, 600, 400, 300, 200, 150 hPa tùy variable | tăng vertical resolution nhưng làm storage và I/O tăng mạnh |
| gần trùng về moisture | total_column_water, total_column_vapor và các biến tích phân liên quan | tránh nhiều cách mã hóa gần cùng một tín hiệu |

Việc chọn 26 channels là hợp lý cho **compact baseline**, nhưng không phải tập tối ưu đã được chứng minh duy nhất. Nó có ba giới hạn cần ghi nhận trung thực:

1. Trạng thái 26 channels không phải một closed physical state đầy đủ; ảnh hưởng của radiation, địa hình, đất–biển và tương tác bề mặt chỉ xuất hiện gián tiếp trong các trường khí quyển đã quan sát.
2. Việc lấy thưa pressure levels có thể làm mất vertical structures mỏng, đặc biệt gần boundary layer, tropopause và jet.
3. Không có static forcing khiến model phải suy ra tác động địa lý từ climatological patterns trong dynamic fields, thay vì được cung cấp trực tiếp.

Do đó cách diễn giải đúng là: **26 channels là điểm khởi đầu có kiểm soát**, đủ đa dạng để học một baseline khí quyển toàn cầu nhưng đủ nhỏ để dự án cá nhân có thể tải, kiểm tra và train. Chỉ nên mở rộng sau khi rollout cho thấy một failure mode cụ thể—chẳng hạn bias bám địa hình/đường bờ, sai chu kỳ ngày hoặc thiếu moisture skill—và khi metric chứng minh channel mới giải quyết đúng failure mode đó.

Đây là **WB2-native approximation** của một compact atmospheric state cho SFNO, không phải channel list nguyên xi từ paper SFNO hay FourCastNet. Archive WeatherBench2 đang dùng không có 100 m winds của thiết kế tham chiếu, nên config chọn specific humidity ở 1000/850 hPa để bổ sung thông tin ẩm tầng thấp.

## 9. Chọn 26 channels có nghĩa là gì đối với dữ liệu tải?

Downloader mở remote Zarr theo kiểu lazy, sau đó chọn đúng:

1. khoảng thời gian đã cấu hình;
2. sáu surface variables;
3. sáu pressure-level variables và đúng các levels trong bảng;
4. các timestep theo stride 6 giờ.

Chỉ sau bước chọn này dữ liệu mới được materialize và ghi ra output. Vì vậy output **không chứa toàn bộ variables hay toàn bộ 13 levels**.

Tuy nhiên, lượng byte thực sự đi qua mạng còn phụ thuộc vào chunk layout của remote Zarr. Nếu một remote chunk chứa nhiều phần tử hơn vùng được chọn, object store vẫn phải gửi cả chunk đó rồi thư viện mới giải nén và lấy phần cần dùng. Hiện tượng này gọi là **read amplification**. Vì vậy cần phân biệt:

- **logical selection:** đúng 26 channels được yêu cầu;
- **network traffic:** có thể lớn hơn kích thước logic do chunking và compression;
- **output storage:** chỉ chứa 26 channels sau regridding.

Nói ngắn gọn: code không chủ động tải mọi feature để lưu lại, nhưng cũng không thể đảm bảo mỗi byte đọc từ cloud đều thuộc đúng một phần tử cuối cùng.

# Phần II — giảm grid từ 0.25° xuống 0.5°

## 10. Kích thước grid trước và sau regridding

| Grid | Latitude | Longitude | Số grid points |
|---|---:|---:|---:|
| nguồn $0.25^\circ$ | 721 | 1440 | 1,038,240 |
| đích $0.5^\circ$ | 361 | 720 | 259,920 |

Số grid points giảm theo tỉ lệ

$$
\frac{721\times1440}{361\times720}\approx3.994.
$$

Tức là gần 4 lần, vì độ phân giải giảm 2 lần trên cả hai trục. Tỉ lệ không đúng 4 tuyệt đối do cả grid nguồn và grid đích đều giữ hai hàng ở hai cực.

Regridding chỉ giảm **spatial resolution**. Nó không bỏ timestep và cũng không bỏ thêm channel nào trong 26 channels đã chọn. Dẫu vậy, chi tiết có wavelength nhỏ hơn khả năng biểu diễn của grid $0.5^\circ$ sẽ không còn nguyên vẹn; đây là phần mất thông tin không thể đảo ngược của mọi phép downsampling.

## 11. Vì sao không lấy cách một điểm?

Cách ngây thơ là giữ mỗi điểm thứ hai theo latitude và longitude. Cách này nhanh nhưng có ba vấn đề:

- các cấu trúc quy mô nhỏ có thể bị alias thành cấu trúc quy mô lớn sai;
- giá trị tại một điểm đơn lẻ không đại diện cho trung bình của ô $0.5^\circ$;
- global mean hoặc area integral của trường có thể thay đổi không kiểm soát.

Bilinear interpolation cũng không phải lựa chọn của dự án. Nó nội suy giá trị tại target point từ các source points gần đó, phù hợp cho nhiều bài toán hiển thị, nhưng không được thiết kế để bảo toàn area integral.

Downloader dùng **first-order conservative area-overlap regridding**. Đây cũng là họ phương pháp WeatherBench2 sử dụng để tạo các bản ERA5 có độ phân giải thấp hơn: xem mỗi source cell có giá trị không đổi và lấy trung bình theo diện tích phần giao với target cell.

## 12. Conservative regridding bằng trực giác

Hãy hình dung target cell $T$ là một ô lớn phủ lên nhiều phần của các source cells $S_i$. Giá trị mới không lấy từ một điểm duy nhất mà là trung bình có trọng số:

$$
x_T=\frac{\sum_i A(T\cap S_i)x_i}{\sum_i A(T\cap S_i)},
$$

trong đó $A(T\cap S_i)$ là diện tích giao giữa target cell và source cell thứ $i$. Source cell phủ target cell nhiều hơn thì đóng góp nhiều hơn.

Vì hai grid được căn tâm tại các bội số của $0.25^\circ$ và $0.5^\circ$, theo longitude một target cell nhận:

$$
\bar x_j=\tfrac14x_{2j-1}+\tfrac12x_{2j}+\tfrac14x_{2j+1}.
$$

Không phải phép lấy trung bình $2\times2$ đơn giản: target cell có tâm trùng một source cell, nên lấy toàn bộ source cell ở giữa và một nửa của mỗi source cell lân cận. Khi chuẩn hóa thành trung bình, trọng số longitude là $0.25, 0.50, 0.25$.

Tại kinh tuyến gốc, chỉ số được wrap theo tính periodic. Vì vậy target cell tâm $0^\circ$ nhận cả đóng góp từ source longitude $359.75^\circ$, thay vì tạo ra một đường nối giả trên bản đồ.

## 13. Trọng số latitude và vai trò của diện tích mặt cầu

Theo latitude, code dựng biên cell từ trung điểm giữa hai cell centres và chặn hai biên ngoài tại $-90^\circ$ và $90^\circ$. Với một dải giao từ $\varphi_a$ đến $\varphi_b$, phần diện tích theo latitude tỉ lệ với

$$
w\propto\sin\varphi_b-\sin\varphi_a.
$$

Các trọng số giao nhau được chuẩn hóa sao cho tổng bằng 1 ở mỗi target latitude. Công thức này xử lý đúng việc các dải latitude gần cực có diện tích nhỏ hơn, thay vì giả định mọi khoảng $0.25^\circ$ đều có cùng diện tích.

Với một trường hằng, output vẫn là đúng hằng số đó. Với trường tổng quát, phương pháp được thiết kế để bảo toàn area-weighted integral đến sai số số học:

$$
\sum_T A_Tx_T\approx\sum_i A_ix_i.
$$

Điều này **không có nghĩa là bảo toàn mọi cực trị địa phương**. Conservative regridding vẫn là phép area averaging, nên peak nhỏ có thể bị làm mượt khi đi từ $0.25^\circ$ xuống $0.5^\circ$.

## 14. Data flow từ archive đến tensor cuối

Luồng biến đổi dữ liệu có thể tóm tắt như sau:

$$
\text{WeatherBench2 Zarr}
\rightarrow\text{chọn time/variables/levels}
\rightarrow\text{conservative regridding}
\rightarrow\text{flatten thành channel}
\rightarrow\text{ghi output Zarr}.
$$

Cụ thể:

1. Mở metadata của remote Zarr, chưa tải toàn bộ array vào RAM.
2. Chọn date range, timestep cadence, variables và pressure levels theo config.
3. Regrid từng field từ $721\times1440$ xuống $361\times720$.
4. Tách từng pressure level thành một channel riêng và ghép theo channel order cố định.
5. Chuyển về layout $[\text{time},\text{channel},\text{latitude},\text{longitude}]$.
6. Lưu float32 theo chunk $[1,26,361,720]$, tức một timestep hoàn chỉnh trên mỗi chunk logic.

Output variable có tên **state**. Các giá trị vẫn mang đơn vị vật lý được ghi trong channel metadata; notebook này không thực hiện data normalization.

# Phần III — kích thước và kiểm chứng dữ liệu

## 15. Kích thước logic của dataset hiện tại

Khoảng thời gian từ 1995-01-01 đến hết 2019-02-16 có 8,813 ngày. Với bốn timestep mỗi ngày:

$$N_t=8813\times4=35{,}252. $$

Tensor đầy đủ có shape

$$[35{,}252,26,361,720].$$

Một timestep float32 sau regridding chiếm

$$26\times361\times720\times4=27{,}031{,}680\text{ bytes}\approx27.03\text{ MB}. $$

Toàn bộ tensor chưa nén chiếm khoảng

$$952.9\text{ GB}=887.5\text{ GiB}. $$

Đây là **logical uncompressed size**, không phải cam kết rằng volume sẽ dùng đúng 952.9 GB. Zarr compression có thể làm dung lượng vật lý nhỏ hơn đáng kể, tùy distribution của từng field, compressor và chunking. Vì vậy phải đo trên một sample đủ đại diện trước khi thuê volume sát giới hạn.

Cùng 26 channels trên source grid $0.25^\circ$ có kích thước chưa nén khoảng 107.98 MB mỗi timestep. Regridding xuống $0.5^\circ$ làm số điểm và logical bytes giảm gần 4 lần.

## 16. Những kiểm tra bắt buộc trước khi train

### 16.1 Schema và coordinates

- state phải có đúng dimension order $[\text{time},\text{channel},\text{latitude},\text{longitude}]$;
- channel count bằng 26 và channel names khớp tuyệt đối bảng ở mục 7;
- grid bằng $361\times720$; longitude chạy từ $0$ đến $359.5^\circ$; latitude bao gồm cả $90^\circ$ và $-90^\circ$;
- timestamp tăng đều đúng 6 giờ, không trùng và không thiếu.

### 16.2 Giá trị vật lý

- không có NaN hoặc infinity ngoài những trường hợp đã được giải thích;
- units trong metadata đúng với source variable;
- min, max, mean và standard deviation theo channel có quy mô hợp lý;
- surface pressure và mean sea-level pressure không bị tráo;
- geopotential không bị hiểu nhầm là geopotential height;
- specific humidity và relative humidity không bị coi là cùng một đại lượng.

### 16.3 Regridding

- trường hằng vẫn hằng sau regrid;
- area-weighted global integral được bảo toàn trong tolerance;
- không xuất hiện seam bất thường tại $0^\circ/360^\circ$ longitude;
- hai hàng cực không chứa giá trị rác hoặc bị nhân đôi sai.

### 16.4 Tính nhất quán khi resume

Downloader lưu fingerprint của data contract gồm source, dates, cadence, channels và regridding version. Khi resume, fingerprint phải khớp; nếu không, cần dùng output path mới thay vì nối hai dataset có ý nghĩa khác nhau.

## 17. Các nhầm lẫn nên tránh

1. **26 channels không có nghĩa là 26 variable names.** Sáu surface variables cộng với 20 lát cắt variable–level tạo thành 26 channels.
2. **Có 13 pressure levels ở archive không có nghĩa là tải cả 13 cho mọi variable.** Mỗi variable có danh sách levels riêng.
3. **Giảm từ 0.25° xuống 0.5° không chỉ giảm 2 lần.** Cả hai trục cùng giảm, nên số grid points giảm gần 4 lần.
4. **Conservative không đồng nghĩa với lossless.** Nó ưu tiên bảo toàn area integral; fine-scale details vẫn bị làm mượt.
5. **Grid cell không có diện tích bằng nhau.** Weight theo latitude là bắt buộc cho các phép tính toàn cầu có ý nghĩa vật lý.
6. **Dung lượng logic không bằng dung lượng file nén và cũng không bằng network traffic.** Ba con số này liên quan nhưng không đồng nhất.
7. **Reanalysis không phải ground truth tuyệt đối.** Nó là ước lượng vật lý nhất quán được tạo bằng forecast model và data assimilation.

## 18. Kết luận

Data contract hiện tại của NeuralAtmosphereOperator là:

$$
\boxed{\text{ERA5/WB2, 6-hourly, 26 channels, }0.5^\circ,\ 361\times720}
$$

Mỗi timestep chứa một trạng thái khí quyển toàn cầu gồm gió, pressure/geopotential, temperature và moisture ở bề mặt cùng nhiều pressure levels. Dataset nguồn $0.25^\circ$ được đưa về $0.5^\circ$ bằng first-order conservative area-overlap regridding có xử lý đúng diện tích mặt cầu và periodic longitude. Không có timestep hay channel nào trong lựa chọn 26 channels bị bỏ trong bước này; phần bị giảm là spatial resolution.

Trước khi train, điều quan trọng nhất không phải chỉ nhìn thấy file Zarr tồn tại, mà phải xác nhận schema, channel order, units, cadence, NaN, periodic seam và conservation tests. Khi toàn bộ các kiểm tra này đạt, output mới có thể được xem là dữ liệu đầu vào đáng tin cậy cho model.

## Tài liệu tham khảo

1. Hersbach, H. et al. (2020), [The ERA5 global reanalysis](https://doi.org/10.1002/qj.3803), *Quarterly Journal of the Royal Meteorological Society*.
2. Rasp, S. et al. (2020), [WeatherBench: A benchmark dataset for data-driven weather forecasting](https://arxiv.org/abs/2002.00469), *Journal of Advances in Modeling Earth Systems*.
3. Rasp, S. et al. (2024), [WeatherBench 2: A benchmark for the next generation of data-driven global weather models](https://arxiv.org/abs/2308.15560), *Journal of Advances in Modeling Earth Systems*.
4. [WeatherBench2 Data Guide](https://weatherbench2.readthedocs.io/en/latest/data-guide.html) — archive paths, native grids, cadence, pressure levels và phương pháp tạo lower-resolution datasets.
5. [ECMWF ERA5 documentation](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) — tổng quan về ERA5 và data assimilation.